# Chapter 1: An Invitation to Geometry

Source orientation: printed pages 1-14; physical PDF pages 11-24; sections 1.1-1.3.

This notebook is a standalone computational lesson. It follows the source chapter's mathematical
order, but the prose, examples, diagrams, and checks are original. The aim is not to reproduce
the book on screen; it is to rebuild the ideas as things a reader can inspect, compute, perturb,
and test. Keep the PDF closed while reading this notebook: every definition needed for the lesson
is introduced here in operational language, and every major claim is paired with a generated
artifact or a numerical sanity check.

**Chapter question.** See geometry as the interplay between local measurements and global shape, using flat tori, spheres, cones, saddles, and Klein's invariant viewpoint.

**Why this chapter matters.** Local measurement, global shape, finite boundaryless surfaces, and the first split between Euclidean, hyperbolic, and elliptic behavior. This course treats geometry as a working laboratory.
When a statement says that an angle is preserved, a boundary is identified, or an area is controlled
by curvature, the notebook asks for a representation that can be drawn and a check that can fail if
the idea has been misunderstood. That habit is the through-line from the complex plane to cosmic
topology.


In [ ]:
# geometry-setup:v1
# Machine-managed by scripts/update_notebook_setup.py. Do not edit this cell by hand.

from __future__ import annotations

import json as _geometry_json
import os as _geometry_os
from pathlib import Path as _GeometryPath
import sys as _geometry_sys

GEOMETRY_SETUP = _geometry_json.loads(
    r"""
{
  "colab_url": "https://colab.research.google.com/github/Rah-Rah-Mitra/Geometry/blob/main/Geometry-with-an-Introduction-to-Cosmic-Topology/chapter-01-an-invitation-to-geometry/01-an-invitation-to-geometry.ipynb",
  "course_dir": "Geometry-with-an-Introduction-to-Cosmic-Topology",
  "course_title": "Geometry with an Introduction to Cosmic Topology",
  "github_url": "https://github.com/Rah-Rah-Mitra/Geometry/blob/main/Geometry-with-an-Introduction-to-Cosmic-Topology/chapter-01-an-invitation-to-geometry/01-an-invitation-to-geometry.ipynb",
  "jupyterlite": false,
  "marker": "geometry-setup:v1",
  "notebook_kind": "lesson",
  "notebook_path": "Geometry-with-an-Introduction-to-Cosmic-Topology/chapter-01-an-invitation-to-geometry/01-an-invitation-to-geometry.ipynb",
  "notebook_title": "Chapter 1: An Invitation to Geometry",
  "repository": {
    "branch": "main",
    "name": "Geometry",
    "owner": "Rah-Rah-Mitra",
    "source_url": "https://github.com/Rah-Rah-Mitra/Geometry"
  },
  "requirements": "requirements/topology.txt",
  "runtime_profile": "topology"
}
"""
)


def _geometry_is_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _geometry_is_jupyterlite():
    return _geometry_sys.platform == "emscripten" or "pyodide" in _geometry_sys.modules


def _geometry_add_path(path):
    text = str(path)
    if text not in _geometry_sys.path:
        _geometry_sys.path.insert(0, text)


def _geometry_find_repo_root():
    candidates = []
    env_root = _geometry_os.environ.get("GEOMETRY_REPO_ROOT")
    if env_root:
        candidates.append(_GeometryPath(env_root).expanduser())
    candidates.append(_GeometryPath.cwd())
    for start in candidates:
        start = start.resolve()
        for current in (start, *start.parents):
            if (current / "course-manifest.json").exists() and (
                current / "metadata" / "runtime_profiles.yml"
            ).exists():
                return current
    raise RuntimeError(
        "Could not find the Geometry repository root. Start JupyterLab inside the "
        "Geometry checkout or set GEOMETRY_REPO_ROOT."
    )


def _geometry_run(command):
    import subprocess as _geometry_subprocess

    printable = " ".join(str(part) for part in command)
    print(f"+ {printable}")
    _geometry_subprocess.check_call([str(part) for part in command])


def _geometry_requirement_names(requirements_path, seen=None):
    seen = set() if seen is None else seen
    requirements_path = requirements_path.resolve()
    if requirements_path in seen or not requirements_path.exists():
        return []
    seen.add(requirements_path)
    names = []
    for raw_line in requirements_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        if line.startswith(("-r ", "--requirement ")):
            _, nested = line.split(maxsplit=1)
            names.extend(_geometry_requirement_names(requirements_path.parent / nested, seen))
            continue
        if line.startswith("-"):
            continue
        name = line
        for separator in ("==", ">=", "<=", "~=", "!=", ">", "<", ";"):
            name = name.split(separator, 1)[0]
        name = name.split("[", 1)[0].strip()
        if name:
            names.append(name)
    return sorted(set(names))


def _geometry_missing_requirements(requirements_path):
    import importlib.metadata as _geometry_metadata

    missing = []
    for name in _geometry_requirement_names(requirements_path):
        try:
            _geometry_metadata.distribution(name)
        except _geometry_metadata.PackageNotFoundError:
            missing.append(name)
    return missing


def _geometry_configured_roots(repo_root):
    course_dir = GEOMETRY_SETUP.get("course_dir")
    course_root = repo_root / course_dir if course_dir else repo_root
    return repo_root, course_root


if _geometry_is_jupyterlite():
    if not GEOMETRY_SETUP["jupyterlite"]:
        raise RuntimeError(
            "This Geometry notebook uses runtime profile "
            f"{GEOMETRY_SETUP['runtime_profile']!r}, which is not enabled for "
            "JupyterLite in course-manifest.json. Open it in Colab or local JupyterLab."
        )
    GEOMETRY_REPO_ROOT = _GeometryPath.cwd()
    GEOMETRY_COURSE_ROOT = (
        GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["course_dir"]
        if GEOMETRY_SETUP.get("course_dir")
        else GEOMETRY_REPO_ROOT
    )
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    if GEOMETRY_COURSE_ROOT.exists():
        _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        "Geometry setup: JupyterLite/Pyodide detected; shell, git, and pip steps "
        "were skipped."
    )
elif _geometry_is_colab():
    repository = GEOMETRY_SETUP["repository"]
    repo_url = repository["source_url"].rstrip("/") + ".git"
    branch = repository["branch"]
    GEOMETRY_REPO_ROOT = _GeometryPath(
        _geometry_os.environ.get("GEOMETRY_REPO_ROOT", "/content/Geometry")
    )
    sparse_paths = ["requirements", "metadata", "scripts", "course-manifest.json", "index.ipynb"]
    if GEOMETRY_SETUP.get("course_dir"):
        sparse_paths.append(GEOMETRY_SETUP["course_dir"])
    if not (GEOMETRY_REPO_ROOT / ".git").exists():
        if GEOMETRY_REPO_ROOT.exists() and any(GEOMETRY_REPO_ROOT.iterdir()):
            raise RuntimeError(
                f"{GEOMETRY_REPO_ROOT} exists but is not a git checkout. "
                "Set GEOMETRY_REPO_ROOT to an empty path or remove the directory."
            )
        _geometry_run(
            [
                "git",
                "clone",
                "--filter=blob:none",
                "--no-checkout",
                "--branch",
                branch,
                repo_url,
                GEOMETRY_REPO_ROOT,
            ]
        )
        _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "init", "--cone"])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "set", *sparse_paths])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "checkout", branch])
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-q", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: Colab ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )
else:
    GEOMETRY_REPO_ROOT = _geometry_find_repo_root()
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    missing = _geometry_missing_requirements(requirements_path)
    skip_install = _geometry_os.environ.get("GEOMETRY_SKIP_INSTALL") == "1"
    if missing and skip_install:
        print(
            "Geometry setup: GEOMETRY_SKIP_INSTALL=1, so missing profile packages "
            f"were not installed: {', '.join(missing)}"
        )
    elif missing:
        print(
            "Geometry setup: installing missing profile packages from "
            f"{requirements_path.relative_to(GEOMETRY_REPO_ROOT)}: {', '.join(missing)}"
        )
        _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: local checkout ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )


## Translation Guide

        The source chapter is translated into computational language as follows:

        - A finite world without boundary becomes computable by replacing edge crossings with equivalence classes.
- A geodesic is represented by the path that a local observer would measure as shortest.
- Triangle angle sum and circle circumference become detectors of curvature.
- The Erlangen viewpoint translates a geometry into a space plus transformations that preserve the chosen measurements.

        A useful mental model is to separate three layers. The **model layer** names the space and its
        coordinates. The **transformation layer** names what is allowed to move without changing the
        geometry. The **measurement layer** names what the inhabitant of the space can detect. A visual
        notebook should keep all three layers on screen. If a curve, point, polygon, or catalog is drawn, the
        nearby text should tell the reader which model it lives in, which transformations are being applied,
        and which measurement is being checked.


## Route Through The Notebook

        - Start from a square video-game universe and unwrap it to a tiled cover.
- Compare Euclidean, spherical, and hyperbolic alternatives to the parallel postulate.
- Use cone and saddle sectors to make angle defect visible.
- End by treating transformations as filters that decide which measurements count as geometric.

        The route is intentionally visual first. We begin each section with a small inspectable construction,
        then attach formulas after the geometry has something to refer to. This is especially important for
        non-Euclidean and quotient-space ideas, where a familiar Euclidean drawing can be correct as a
        coordinate picture but misleading as a metric picture.


In [ ]:
from pathlib import Path
import sys

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the GICT book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / "chapter-01"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

CHAPTER_META = {
    "kind": "chapter",
    "number": 1,
    "title": "An Invitation to Geometry",
    "printed": "1-14",
    "pdf": "11-24",
    "sections": "1.1-1.3",
    "goal": "See geometry as the interplay between local measurements and global shape, using flat tori, spheres, cones, saddles, and Klein's invariant viewpoint.",
    "focus": "Local measurement, global shape, finite boundaryless surfaces, and the first split between Euclidean, hyperbolic, and elliptic behavior."
}
VISUAL_SPECS = [
    {
        "kind": "torus",
        "filename": "flat-torus-edge-identification.png",
        "title": "Flat torus edge identification",
        "note": "Opposite edges are the same place, not walls."
    },
    {
        "kind": "sphere",
        "filename": "sphere-great-circle-triangle.png",
        "title": "Spherical triangle angle excess",
        "note": "Great-circle edges make a triangle whose angle sum exceeds pi."
    },
    {
        "kind": "parallel",
        "filename": "parallel-postulate-comparison.png",
        "title": "Three parallel-postulate worlds",
        "note": "One, many, or zero parallels through a point."
    },
    {
        "kind": "cone",
        "filename": "cone-saddle-angle-defect.png",
        "title": "Cone and saddle sector defect",
        "note": "Sector angle controls circumference and triangle angle sum."
    },
    {
        "kind": "hexagon",
        "filename": "hexagon-edge-gluing.png",
        "title": "Hexagon edge gluing",
        "note": "Corner totals explain when a glued polygon is homogeneous."
    },
    {
        "kind": "erlangen",
        "filename": "erlangen-invariant-filter.png",
        "title": "Erlangen invariant filter",
        "note": "The allowed transformations decide which features survive."
    }
]

from utils.artifacts import assert_artifacts, display_artifact, save_json, save_table, save_html
from utils.visuals import build_parameter_lab_html, chapter_numeric_checks, render_visuals


## Visual Storyboard

        The generated artifacts for this notebook are:

        - **Flat torus edge identification** (`flat-torus-edge-identification.png`): Opposite edges are the same place, not walls.
- **Spherical triangle angle excess** (`sphere-great-circle-triangle.png`): Great-circle edges make a triangle whose angle sum exceeds pi.
- **Three parallel-postulate worlds** (`parallel-postulate-comparison.png`): One, many, or zero parallels through a point.
- **Cone and saddle sector defect** (`cone-saddle-angle-defect.png`): Sector angle controls circumference and triangle angle sum.
- **Hexagon edge gluing** (`hexagon-edge-gluing.png`): Corner totals explain when a glued polygon is homogeneous.
- **Erlangen invariant filter** (`erlangen-invariant-filter.png`): The allowed transformations decide which features survive.

        Each filename names the concept being inspected. The final sanity cell asserts that the artifacts
        exist, are nonempty, and are visually nonblank. That check is deliberately prosaic: if the course is
        going to make visual explanation central, blank or decorative figures must be treated as failures,
        not as cosmetic issues.

        Read the figures as a sequence rather than as isolated illustrations. The first visual usually fixes
        the model: a plane, disk, sphere, polygon, quotient domain, or catalog. The middle visuals change
        one parameter or one transformation at a time so that the invariant has a chance to stand out. The
        last visuals connect the computation back to the chapter's larger question. A useful habit is to ask
        three questions after every display: What object is being represented? Which measurement is being
        preserved or changed? Which part of the code would have to be wrong for the picture to lie?

        The artifacts also make the notebook reproducible. The JSON files record small residuals and
        counts, the CSV tables record the visual inventory, and the HTML lab gives a low-friction place to
        perturb a parameter. None of these files replace mathematical proof, but they make the proof goals
        more concrete. For example, a theorem about angle preservation becomes easier to parse after the
        reader has watched distances change while angle checks remain stable; a classification statement
        becomes less abstract after a table of invariants separates examples that initially look alike.

        When adapting this notebook, keep the same standard: every new visual should earn its place by
        revealing a construction, invariant, obstruction, or failure mode. A pretty picture with no inspection
        target should be removed or rewritten until the reader knows what to look for.


In [ ]:
figure_paths, visual_stats = render_visuals(ARTIFACT_ROOT, VISUAL_SPECS)
for path in figure_paths:
    display_artifact(path, width=820)
visual_stats


## Worked Example

A recurring worked example in this chapter is to choose a simple test object, transform it, and ask
what changed. For a complex vector this might mean comparing modulus and argument before and
after multiplication. For a hyperbolic geodesic it might mean checking that the Euclidean circle meets
the disk boundary orthogonally. For a surface quotient it might mean walking across an edge and
returning through its identified partner. The particular objects differ from chapter to chapter, but the
discipline is the same: draw the object, compute a small invariant, perturb a parameter, and explain
which observation survives.

The code below follows that pattern. It creates the visual artifacts from a compact storyboard, then
writes a JSON check file and a CSV inventory. The checks are not a proof of every theorem in the
chapter; they are executable guardrails that keep the main constructions honest.


In [ ]:
checks = chapter_numeric_checks(int(CHAPTER_META["number"]))
checks["visual_count"] = len(figure_paths)
checks["minimum_pixel_std"] = min(item["pixel_std"] for item in visual_stats)
checks_path = save_json(checks, ARTIFACT_ROOT, "checks", "chapter-checks.json")
display_artifact(checks_path)
checks


## Applied Lab

Build a small universe simulator: choose a rectangle, place an observer and an object, then compute the nearest image in the tiled cover.

Treat the lab as an invitation to change parameters rather than as a fixed exercise. The important
output is a comparison: what stays invariant, what changes smoothly, and what breaks when the
assumptions are violated? A strong answer includes a picture, a numerical residual, and one sentence
explaining why the residual is the right thing to measure.


In [ ]:
html_path = save_html(build_parameter_lab_html(CHAPTER_META, VISUAL_SPECS), ARTIFACT_ROOT, "html", "parameter-lab.html")
display_artifact(html_path, height=420)
html_path


In [ ]:
inventory_rows = [
    {
        "artifact": path.name,
        "category": path.parent.name,
        "bytes": path.stat().st_size,
        "chapter": CHAPTER_META["title"],
    }
    for path in figure_paths
]
table_path = save_table(inventory_rows, ARTIFACT_ROOT, "tables", "visual-inventory.csv")
display_artifact(table_path)
inventory_rows[:2]


## Pitfalls

        - The embedded donut in three-dimensional space is not the same geometry as the flat torus quotient.
- A drawn curve on a surface is not automatically a geodesic.
- A finite universe does not need an edge; it can close through identification.

        These pitfalls are worth naming because the pictures in this subject are powerful enough to mislead.
        A Euclidean-looking disk can carry a hyperbolic metric. A boundary of a polygon may be an
        identification pattern rather than a wall. A local measurement can be insensitive to global topology.
        The notebooks keep these distinctions explicit by pairing drawings with formulas and by saving the
        intermediate checks.


In [ ]:
all_paths = list(figure_paths) + [checks_path, html_path, table_path]
assert_artifacts(all_paths)
assert len(figure_paths) >= 5
assert min(item["pixel_std"] for item in visual_stats) > 2.0
assert checks["status"] == "ok"
final_sanity = {
    "artifact_count": len(all_paths),
    "min_pixel_std": min(item["pixel_std"] for item in visual_stats),
    "all_paths_exist": all(path.exists() for path in all_paths),
}
final_path = save_json(final_sanity, ARTIFACT_ROOT, "checks", "final-sanity.json")
display_artifact(final_path)
final_sanity


## Takeaways

- The chapter's central ideas can be inspected through generated diagrams rather than memorized from static figures.
- The relevant formulas act as checks on the visuals: distance, angle, area, orbit, or topology data should agree with the drawing.
- A good model separates coordinate appearance from intrinsic measurement.
- The artifacts are part of the course, not byproducts; rerunning the notebook rebuilds them from the local utilities.

Before moving on, choose one artifact and explain it without using the textbook's wording: name the
model, name the transformation or measurement, and name the visible evidence. If that explanation
feels vague, rerun the relevant cell with a changed parameter and watch what remains stable.
